In [7]:
import joblib
from pycaret.classification import *
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, RepeatedKFold, StratifiedKFold, train_test_split, GridSearchCV, RandomizedSearchCV
from tsfresh import extract_features, select_features
import shap
import os
import matplotlib.pyplot as plt

In [ ]:
#CRF WITH STRATA
# CRF RANDOM FOREST MODEL

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Prepare CRF data

CRF_matrix = pd.read_csv("./CRF_matrix.txt",sep="\t")

CRF_matrix = CRF_matrix.set_index('SUBJECT_ID')

CRF_Y = CRF_matrix['PHENOTYPE']

CRF_X = CRF_matrix.drop(
    columns=['PHENOTYPE']
)

# 2. Keep only phenotype values 1 and 2

val = CRF_Y[
    ~CRF_Y.isin([1, 2])
].index.tolist()

CRF_Y = CRF_Y.drop(index=val)
CRF_X = CRF_X.drop(index=val)
CRF_matrix = CRF_matrix.drop(index=val)


# 3. Convert phenotype
#
# 1 = control -> 0
# 2 = case    -> 1

CRF_Y = CRF_Y.astype(int) - 1


# 4. Fill missing values

CRF_X = CRF_X.fillna(0)


# 5. Get ancestry from PRS matrix
#
# ancestry is ONLY used for stratification.
# It is NOT included as a CRF feature.

PRS_ancestry = PRS_matrix['ancestry']


# Keep only CRF subjects that have ancestry information
CRF_ancestry = PRS_ancestry.reindex(CRF_X.index)


# Check for missing ancestry
print("\nMISSING CRF ANCESTRY")
print("--------------------")
print(CRF_ancestry.isna().sum())


# 6. Create ancestry + phenotype strata

CRF_strata = (
    CRF_ancestry.astype(str)
    + '_'
    + CRF_Y.astype(str)
)


# 7. FIRST SPLIT
# 80% development
# 20% unseen test

X_trainCRF, X_testCRF, \
Y_trainCRF, Y_testCRF = train_test_split(
    CRF_X,
    CRF_Y,
    test_size=0.2,
    random_state=10,
    stratify=CRF_strata
)


# 8. Stratification variable for second split

CRF_strata_development = (
    CRF_ancestry.loc[X_trainCRF.index].astype(str)
    + '_'
    + Y_trainCRF.astype(str)
)


# 9. SECOND SPLIT
#
# 70% of development -> training
# 30% of development -> validation
#
# Overall:
# 56% training
# 24% validation
# 20% unseen test

X_train2CRF, X_test2CRF, \
Y_train2CRF, Y_test2CRF = train_test_split(
    X_trainCRF,
    Y_trainCRF,
    test_size=0.3,
    random_state=10,
    stratify=CRF_strata_development
)


# 10. Create datasets

trainingCRF = X_train2CRF.copy()
trainingCRF['PHENOTYPE'] = Y_train2CRF

testCRF = X_test2CRF.copy()
testCRF['PHENOTYPE'] = Y_test2CRF

unseen_testCRF = X_testCRF.copy()
unseen_testCRF['PHENOTYPE'] = Y_testCRF


# 11. Check splits

print("\nCRF SPLIT SIZES")
print("---------------")

print("Training:", len(trainingCRF))
print("Validation:", len(testCRF))
print("Unseen test:", len(unseen_testCRF))


print("\nCRF TRAINING DISTRIBUTION")
print("-------------------------")

print(
    pd.crosstab(
        CRF_ancestry.loc[trainingCRF.index],
        trainingCRF['PHENOTYPE']
    )
)


print("\nCRF VALIDATION DISTRIBUTION")
print("---------------------------")

print(
    pd.crosstab(
        CRF_ancestry.loc[testCRF.index],
        testCRF['PHENOTYPE']
    )
)


print("\nCRF UNSEEN TEST DISTRIBUTION")
print("----------------------------")

print(
    pd.crosstab(
        CRF_ancestry.loc[unseen_testCRF.index],
        unseen_testCRF['PHENOTYPE']
    )
)


# 12. PyCaret

CRF = ClassificationExperiment()

CRF.setup(
    trainingCRF,
    target='PHENOTYPE',
    session_id=10,
    test_data=testCRF
)


# 13. Random Forest

rfCRF = CRF.create_model(
    'rf',
    class_weight='balanced'
)

# 14. Tune for F1

tuned_rfCRF = CRF.tune_model(
    rfCRF,
    optimize='F1'
)


# 15. Predictions

pred_CRF = CRF.predict_model(
    tuned_rfCRF
)


# 16. Predictions on unseen test set

pred_test_CRF = CRF.predict_model(
    tuned_rfCRF,
    data=unseen_testCRF
)

In [ ]:
# QTD RANDOM FOREST MODEL
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Prepare QTD data

qtd_matrix = pd.read_csv(
    './qtd_matrix.txt',
    delimiter='\t'
)

qtd_matrix = qtd_matrix.rename(
    columns={
        'Vessel_like_ml': 'Vessel-like volume',
        'QtD_Full': 'QtD Full',
        'QtD_MD': 'QtD MD',
        'QtD_LD': 'QtD LD'
    }
)

qtd_matrix = qtd_matrix.set_index(
    "SUBJECT_ID"
)

qtd_Y = qtd_matrix['PHENOTYPE']

qtd_X = qtd_matrix.drop(
    columns=['sid', 'PHENOTYPE']
)


# 2. Keep only phenotype values 1 and 2

val = qtd_Y[
    ~qtd_Y.isin([1, 2])
].index.tolist()

qtd_Y = qtd_Y.drop(index=val)
qtd_X = qtd_X.drop(index=val)
qtd_matrix = qtd_matrix.drop(index=val)

# 3. Convert phenotype
#
# 1 = control -> 0
# 2 = case    -> 1

qtd_Y = qtd_Y.astype(int) - 1

      
# 4. Fill missing values


qtd_X = qtd_X.fillna(0)

# 5. Get ancestry from PRS matrix
#
# ancestry is ONLY used for stratification.
# It is NOT included as a QTD feature.

PRS_ancestry = PRS_matrix['ancestry']

qtd_ancestry = PRS_ancestry.reindex(
    qtd_X.index
)

# Check for missing ancestry
print("\nMISSING QTD ANCESTRY")
print("--------------------")
print(qtd_ancestry.isna().sum())

# 6. Create ancestry + phenotype strata

qtd_strata = (
    qtd_ancestry.astype(str)
    + '_'
    + qtd_Y.astype(str)
)

# 7. FIRST SPLIT
#
# 80% development
# 20% unseen test

X_trainQ, X_testQ, \
Y_trainQ, Y_testQ = train_test_split(
    qtd_X,
    qtd_Y,
    test_size=0.2,
    random_state=10,
    stratify=qtd_strata
)

# 8. Stratification variable for second split
qtd_strata_development = (
    qtd_ancestry.loc[X_trainQ.index].astype(str)
    + '_'
    + Y_trainQ.astype(str)
)

# 9. SECOND SPLIT
#
# 70% of development -> training
# 30% of development -> validation
#
# Overall:
# 56% training
# 24% validation
# 20% unseen test

X_train2Q, X_test2Q, \
Y_train2Q, Y_test2Q = train_test_split(
    X_trainQ,
    Y_trainQ,
    test_size=0.3,
    random_state=10,
    stratify=qtd_strata_development
)

# 10. Create datasets
trainingQ = X_train2Q.copy()
trainingQ['PHENOTYPE'] = Y_train2Q

testQ = X_test2Q.copy()
testQ['PHENOTYPE'] = Y_test2Q

unseen_testQ = X_testQ.copy()
unseen_testQ['PHENOTYPE'] = Y_testQ

# 11. Check splits
print("\nQTD SPLIT SIZES")
print("---------------")

print("Training:", len(trainingQ))
print("Validation:", len(testQ))
print("Unseen test:", len(unseen_testQ))


print("\nQTD TRAINING DISTRIBUTION")
print("-------------------------")

print(
    pd.crosstab(
        qtd_ancestry.loc[trainingQ.index],
        trainingQ['PHENOTYPE']
    )
)


print("\nQTD VALIDATION DISTRIBUTION")
print("---------------------------")

print(
    pd.crosstab(
        qtd_ancestry.loc[testQ.index],
        testQ['PHENOTYPE']
    )
)


print("\nQTD UNSEEN TEST DISTRIBUTION")
print("----------------------------")

print(
    pd.crosstab(
        qtd_ancestry.loc[unseen_testQ.index],
        unseen_testQ['PHENOTYPE']
    )
)



# 12. PyCaret

qtd = ClassificationExperiment()

qtd.setup(
    trainingQ,
    target='PHENOTYPE',
    session_id=10,
    test_data=testQ
)

# 13. Random Forest

rfQ = qtd.create_model(
    'rf',
    class_weight='balanced'
)

# 14. Tune for F1

tuned_rfQ = qtd.tune_model(
    rfQ,
    optimize='F1'
)


# 15. Predictions

pred_QTD = qtd.predict_model(
    tuned_rfQ
)
# 16. Predictions on unseen test set

pred_test_qtd = qtd.predict_model(
    tuned_rfQ,
    data=unseen_testQ
)

In [ ]:
# PRS RANDOM FOREST MODEL PRS WITH STRATA

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    brier_score_loss
)

from sklearn.calibration import calibration_curve


# 1. Prepare PRS data

PRS_matrix = pd.read_csv(
    "./PRS_matrix.txt",
    sep="\t"
)

PRS_matrix = PRS_matrix.set_index('SUBJECT_ID')

PRS_Y = PRS_matrix['PHENOTYPE']

PRS_X = PRS_matrix.drop(
    columns=['PHENOTYPE']
)

# 2. Keep only phenotype values 1 and 2

PRS_invalid_index = PRS_Y[
    ~PRS_Y.isin([1, 2])
].index.tolist()

PRS_Y = PRS_Y.drop(
    index=PRS_invalid_index
)

PRS_X = PRS_X.drop(
    index=PRS_invalid_index
)

PRS_matrix = PRS_matrix.drop(
    index=PRS_invalid_index
)

# 3. Convert phenotype
#
# 1 = control -> 0
# 2 = case    -> 1

PRS_Y = PRS_Y.astype(int) - 1

# 4. Fill missing values

PRS_X = PRS_X.fillna(0)

# 5. Create complete dataset

PRS_data = PRS_X.copy()

PRS_data['PHENOTYPE'] = PRS_Y


# 6. Check original distribution

print("\nORIGINAL DATA")
print("-------------")

print(
    pd.crosstab(
        PRS_data['ancestry'],
        PRS_data['PHENOTYPE']
    )
)

# 7. Separate X and Y

PRS_Y = PRS_data['PHENOTYPE']

PRS_X = PRS_data.drop(
    columns=['PHENOTYPE']
)


# 8. Stratify by ancestry + phenotype

PRS_strata = (
    PRS_X['ancestry'].astype(str)
    + '_'
    + PRS_Y.astype(str)
)

# 9. FIRST SPLIT
#
# 80% development
# 20% unseen test

PRS_X_development, PRS_X_unseen_test, \
PRS_Y_development, PRS_Y_unseen_test = train_test_split(
    PRS_X,
    PRS_Y,
    test_size=0.20,
    random_state=10,
    stratify=PRS_strata
)


# 10. Stratification variable for second split

PRS_strata_development = (
    PRS_X_development['ancestry'].astype(str)
    + '_'
    + PRS_Y_development.astype(str)
)

# 11. SECOND SPLIT
#
# 56% training
# 24% validation
# 20% unseen test

PRS_X_train, PRS_X_validation, \
PRS_Y_train, PRS_Y_validation = train_test_split(
    PRS_X_development,
    PRS_Y_development,
    test_size=0.30,
    random_state=10,
    stratify=PRS_strata_development
)

# 12. Create datasets

PRS_training = PRS_X_train.copy()

PRS_training['PHENOTYPE'] = PRS_Y_train


PRS_validation = PRS_X_validation.copy()

PRS_validation['PHENOTYPE'] = PRS_Y_validation


PRS_unseen_test = PRS_X_unseen_test.copy()

PRS_unseen_test['PHENOTYPE'] = PRS_Y_unseen_test

# 13. Check training distribution

print("\nTRAINING DATA")
print("-------------")

print(
    pd.crosstab(
        PRS_training['ancestry'],
        PRS_training['PHENOTYPE']
    )
)


# 14. Check validation distribution

print("\nVALIDATION DATA")
print("----------------")

print(
    pd.crosstab(
        PRS_validation['ancestry'],
        PRS_validation['PHENOTYPE']
    )
)


# 15. Check unseen test distribution

print("\nUNSEEN TEST DATA")
print("----------------")

print(
    pd.crosstab(
        PRS_unseen_test['ancestry'],
        PRS_unseen_test['PHENOTYPE']
    )
)


# 16. Calculate AA case weight
#
# AA case weight =
#
# number of AA controls
# ---------------------
#   number of AA cases
#
# IMPORTANT:
# Calculated using TRAINING DATA ONLY.

PRS_AA_cases_train = (
    (PRS_training['ancestry'] == 2) &
    (PRS_training['PHENOTYPE'] == 1)
).sum()


PRS_AA_controls_train = (
    (PRS_training['ancestry'] == 2) &
    (PRS_training['PHENOTYPE'] == 0)
).sum()


PRS_AA_case_weight = (
    PRS_AA_controls_train /
    PRS_AA_cases_train
)


print("\nAA SAMPLE WEIGHTING")
print("-------------------")

print(
    "AA controls in training:",
    PRS_AA_controls_train
)

print(
    "AA cases in training:",
    PRS_AA_cases_train
)

print(
    "AA case weight:",
    PRS_AA_case_weight
)


# 17. Create sample weights
#
# Everyone starts with weight = 1
#
# AA cases receive:
#     AA controls / AA cases
#
# AA controls = 1
# NHW controls = 1
# NHW cases = 1

PRS_sample_weights = np.ones(
    len(PRS_training)
)


PRS_AA_case_mask = (
    (PRS_training['ancestry'] == 2) &
    (PRS_training['PHENOTYPE'] == 1)
)


PRS_sample_weights[
    PRS_AA_case_mask.values
] = PRS_AA_case_weight


# 18. Check weighted training contribution

PRS_weighted_counts = pd.DataFrame({

    'ancestry':
        PRS_training['ancestry'].values,

    'phenotype':
        PRS_training['PHENOTYPE'].values,

    'weight':
        PRS_sample_weights

})


PRS_weighted_summary = (
    PRS_weighted_counts
    .groupby(
        ['ancestry', 'phenotype']
    )['weight']
    .sum()
)


print("\nWEIGHTED TRAINING CONTRIBUTION")
print("------------------------------")

print(
    PRS_weighted_summary
)


# 19. Prepare X and Y for sklearn

PRS_X_train = PRS_training.drop(
    columns=['PHENOTYPE']
)

PRS_y_train = PRS_training[
    'PHENOTYPE'
].astype(int)


PRS_X_validation = PRS_validation.drop(
    columns=['PHENOTYPE']
)

PRS_y_validation = PRS_validation[
    'PHENOTYPE'
].astype(int)


PRS_X_unseen_test = PRS_unseen_test.drop(
    columns=['PHENOTYPE']
)

PRS_y_unseen_test = PRS_unseen_test[
    'PHENOTYPE'
].astype(int)

# 20. Untuned Random Forest

PRS_rf = RandomForestClassifier(
    n_estimators=500,
    random_state=10,
    n_jobs=-1
)


# 21. Train using sample weights

PRS_rf.fit(
    PRS_X_train,
    PRS_y_train,
    sample_weight=PRS_sample_weights
)

# 22. Predict probabilities on validation set

#specifically probability of class 1 (case)
PRS_prob_val = PRS_rf.predict_proba(
    PRS_X_validation
)[:, 1]


# 23. Standard 0.5 threshold - validation set

PRS_pred_val = (
    PRS_prob_val >= 0.5
).astype(int)


# 24. Predict probabilities on unseen test set

PRS_prob_test = PRS_rf.predict_proba(
    PRS_X_unseen_test
)[:, 1]


# 25. Standard 0.5 threshold - unseen test set

PRS_pred_test = (
    PRS_prob_test >= 0.5
).astype(int)


# 26. Create results dataframe

PRS_results_test = PRS_unseen_test.copy()

PRS_results_test['y_true'] = (
    PRS_y_unseen_test.values
)

PRS_results_test['probability'] = (
    PRS_prob_test
)

PRS_results_test['prediction'] = (
    PRS_pred_test
)


# 27. Calculate metrics
#
# Overall + NHW + AA

PRS_results_summary = []


# Define groups

PRS_groups = [

    (
        'Overall',
        PRS_results_test
    ),

    (
        'NHW',
        PRS_results_test[
            PRS_results_test['ancestry'] == 1
        ]
    ),

    (
        'AA',
        PRS_results_test[
            PRS_results_test['ancestry'] == 2
        ]
    )

]

# Calculate metrics for each group

for PRS_group_name, PRS_subset in PRS_groups:

    PRS_y_true = PRS_subset[
        'y_true'
    ].values

    PRS_y_pred = PRS_subset[
        'prediction'
    ].values

    PRS_y_prob = PRS_subset[
        'probability'
    ].values


    # Sample counts
    PRS_n_total = len(PRS_subset)

    PRS_n_controls = np.sum(
        PRS_y_true == 0
    )

    PRS_n_cases = np.sum(
        PRS_y_true == 1
    )


    # AUC

    if len(np.unique(PRS_y_true)) == 2:

        PRS_auc = roc_auc_score(
            PRS_y_true,
            PRS_y_prob
        )

    else:

        PRS_auc = np.nan


    # Confusion matrix

    PRS_cm = confusion_matrix(
        PRS_y_true,
        PRS_y_pred,
        labels=[0, 1]
    )

    PRS_tn, PRS_fp, PRS_fn, PRS_tp = PRS_cm.ravel()


    # Accuracy

    PRS_accuracy = accuracy_score(
        PRS_y_true,
        PRS_y_pred
    )


    # Sensitivity

    PRS_sensitivity = recall_score(
        PRS_y_true,
        PRS_y_pred,
        zero_division=0
    )


    # Specificity

    PRS_specificity = (
        PRS_tn / (PRS_tn + PRS_fp)
        if (PRS_tn + PRS_fp) > 0
        else np.nan
    )


    # PPV

    PRS_ppv = precision_score(
        PRS_y_true,
        PRS_y_pred,
        zero_division=0
    )

    # NPV

    PRS_npv = (
        PRS_tn / (PRS_tn + PRS_fn)
        if (PRS_tn + PRS_fn) > 0
        else np.nan
    )


    # F1

    PRS_f1 = f1_score(
        PRS_y_true,
        PRS_y_pred,
        zero_division=0
    )


    # Brier score

    PRS_brier = brier_score_loss(
        PRS_y_true,
        PRS_y_prob
    )


    # Store results

    PRS_results_summary.append({

        'Group': PRS_group_name,

        'N': PRS_n_total,

        'Controls': PRS_n_controls,

        'Cases': PRS_n_cases,

        'Threshold': 0.5,

        'AUC': PRS_auc,

        'Accuracy': PRS_accuracy,

        'Sensitivity': PRS_sensitivity,

        'Specificity': PRS_specificity,

        'PPV': PRS_ppv,

        'NPV': PRS_npv,

        'F1': PRS_f1,

        'Brier': PRS_brier,

        'TN': PRS_tn,

        'FP': PRS_fp,

        'FN': PRS_fn,

        'TP': PRS_tp

    })


# 28. Final results table

PRS_results_df = pd.DataFrame(
    PRS_results_summary
)


print("\nFINAL PRS RESULTS")
print("-----------------")

print(
    PRS_results_df.to_string(
        index=False
    )
)


# 29. Calibration plot
#
# Overall + NHW + AA

plt.figure(
    figsize=(6, 6)
)


for PRS_group_name, PRS_subset in [

    (
        'Overall',
        PRS_results_test
    ),

    (
        'NHW',
        PRS_results_test[
            PRS_results_test['ancestry'] == 1
        ]
    ),

    (
        'AA',
        PRS_results_test[
            PRS_results_test['ancestry'] == 2
        ]
    )

]:

    PRS_y_true = PRS_subset[
        'y_true'
    ].values

    PRS_y_prob = PRS_subset[
        'probability'
    ].values

    # Calibration curve

    PRS_prob_true, PRS_prob_pred = calibration_curve(
        PRS_y_true,
        PRS_y_prob,
        n_bins=10,
        strategy='quantile'
    )


    # Plot calibration curve

    plt.plot(
        PRS_prob_pred,
        PRS_prob_true,
        marker='o',
        linewidth=2,
        label=PRS_group_name
    )

# 30. Perfect calibration line

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    linewidth=1.5,
    label='Perfect calibration'
)


# 31. Formatting

plt.xlabel(
    'Mean predicted probability'
)

plt.ylabel(
    'Observed proportion of cases'
)

plt.title(
    'Calibration Plot – PRS Random Forest\n'
    'AA Case Sample Weighting'
)

plt.xlim(
    0,
    1
)

plt.ylim(
    0,
    1
)

plt.legend(
    loc='lower right'
)

plt.grid(
    alpha=0.2
)

plt.tight_layout()

plt.show()

In [ ]:
#Ensemble

In [ ]:
# ENSEMBLE 
# CRF/QTD:
# prediction_score = probability of predicted class
#
# PRS:
# converted from P(case) to probability of predicted class

# VALIDATION SET

comb_pred = pd.DataFrame()

# CRF - PyCaret
# prediction_score = probability of predicted class


comb_pred['CRF'] = pred_CRF['prediction_score']

comb_pred.loc[
    pred_CRF['prediction_label'] == 0,
    'CRF'
] *= -1


# PRS - sklearn
# PRS_pred_val, array of 0 and 1 for conts and cases
# PRS_prob_val = probability of CASE (class 1)
#
# Convert this to probability of the PREDICTED class:
#
# if prediction = 1:
#     probability = P(case)
#
# if prediction = 0:
#     probability = P(control) = 1 - P(case)


PRS_pred_class_prob_val = np.where(
    PRS_pred_val == 1,
    PRS_prob_val,
    1 - PRS_prob_val
)

# Sign flip controls

comb_pred['PRS'] = PRS_pred_class_prob_val

comb_pred.loc[
    PRS_pred_val == 0,
    'PRS'
] *= -1


# QTD
# prediction_score = probability of predicted class


comb_pred['QTD_BV'] = pred_QTD['prediction_score']

comb_pred.loc[
    pred_QTD['prediction_label'] == 0,
    'QTD_BV'
] *= -1


# PHENOTYPE


comb_pred['PHENOTYPE'] = pred_CRF['PHENOTYPE']


# CHECK VALIDATION ENSEMBLE


print("\nVALIDATION ENSEMBLE")
print("-------------------")

print(comb_pred.head())

print("\nClass counts:")
print(comb_pred['PHENOTYPE'].value_counts())


# UNSEEN TEST SET


comb_test = pd.DataFrame()


# CRF
# prediction_score = probability of predicted class


comb_test['CRF'] = pred_test_CRF['prediction_score']

comb_test.loc[
    pred_test_CRF['prediction_label'] == 0,
    'CRF'
] *= -1


# PRS 
# PRS_prob_test = probability of CASE (class 1)
# Convert to probability of PREDICTED class


PRS_pred_class_prob_test = np.where(
    PRS_pred_test == 1,
    PRS_prob_test,
    1 - PRS_prob_test
)


comb_test['PRS'] = PRS_pred_class_prob_test

comb_test.loc[
    PRS_pred_test == 0,
    'PRS'
] *= -1


# QTD - PyCaret
#
# prediction_score = probability of predicted class


comb_test['QTD_BV'] = pred_test_qtd['prediction_score']

comb_test.loc[
    pred_test_qtd['prediction_label'] == 0,
    'QTD_BV'
] *= -1


# PHENOTYPE


comb_test['PHENOTYPE'] = pred_test_CRF['PHENOTYPE']


# CHECK TEST ENSEMBLE

print("\nUNSEEN TEST ENSEMBLE")
print("--------------------")

print(comb_test.head())

print("\nClass counts:")
print(comb_test['PHENOTYPE'].value_counts())


# CHECK SCORE RANGES

print("\nSCORE RANGES")
print("------------")

print(
    "CRF:",
    comb_test['CRF'].min(),
    "to",
    comb_test['CRF'].max()
)

print(
    "PRS:",
    comb_test['PRS'].min(),
    "to",
    comb_test['PRS'].max()
)

print(
    "QTD:",
    comb_test['QTD_BV'].min(),
    "to",
    comb_test['QTD_BV'].max()
)


# PYCARET ENSEMBLE

comb = ClassificationExperiment()

comb.setup(
    comb_pred,
    target='PHENOTYPE',
    session_id=10,
    test_data=comb_test,
    index=False
)


# RANDOM FOREST ENSEMBLE

rfC = comb.create_model(
    'rf'
)

# TUNE FOR F1

tuned_rfC = comb.tune_model(
    rfC,
    optimize='F1'
)

# PREDICT ON UNSEEN TEST SET

predC = comb.predict_model(
    tuned_rfC
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# ============================================================
# ROC CURVES - ALL MODELS
# ============================================================

# CRF
fpr_CRF, tpr_CRF, _ = roc_curve(
    y_test,
    CRF_test_probability_case
)

auc_CRF = roc_auc_score(
    y_test,
    CRF_test_probability_case
)


# PRS
fpr_PRS, tpr_PRS, _ = roc_curve(
    y_test,
    PRS_test_probability_case
)

auc_PRS = roc_auc_score(
    y_test,
    PRS_test_probability_case
)


# Imaging
fpr_Imaging, tpr_Imaging, _ = roc_curve(
    y_test,
    QTD_test_probability_case
)

auc_Imaging = roc_auc_score(
    y_test,
    QTD_test_probability_case
)


# Ensemble
fpr_Ensemble, tpr_Ensemble, _ = roc_curve(
    y_test,
    ensemble_test_probability_case
)

auc_Ensemble = roc_auc_score(
    y_test,
    ensemble_test_probability_case
)


# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(7, 6))

plt.plot(
    fpr_CRF,
    tpr_CRF,
    linewidth=2,
    label=f'CRF (AUC = {auc_CRF:.3f})'
)

plt.plot(
    fpr_PRS,
    tpr_PRS,
    linewidth=2,
    label=f'PRS (AUC = {auc_PRS:.3f})'
)

plt.plot(
    fpr_Imaging,
    tpr_Imaging,
    linewidth=2,
    label=f'Imaging (AUC = {auc_Imaging:.3f})'
)

plt.plot(
    fpr_Ensemble,
    tpr_Ensemble,
    linewidth=2.5,
    label=f'Ensemble (AUC = {auc_Ensemble:.3f})'
)


# Random classifier
plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    linewidth=1.5,
    label='Chance'
)


# ============================================================
# FORMATTING
# ============================================================

plt.xlabel('False Positive Rate (1 − Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')

plt.title(
    'ROC Curves – Unseen Test Set'
)

plt.xlim(0, 1)
plt.ylim(0, 1)

plt.legend(
    loc='lower right'
)

plt.grid(
    alpha=0.2
)

plt.tight_layout()
plt.savefig("./ROCcures_combined.pdf", format="pdf")
plt.show()